# Automatically find all person names in your reference lists and paragraphs and save them with their source file and volume information.


#### 1. Imports

I decided for spacy [because](https://www.geeksforgeeks.org/python/python-named-entity-recognition-ner-using-spacy/): 

In [ ]:
import pandas as pd
import spacy

# run this once in terminal if you haven't already:
# python -m spacy download en_core_web_sm
nlp = spacy.load("en_core_web_sm")

#### 2. Loading the H2 Corpus

In [6]:
df = pd.read_csv("/Users/sophiehamann/master-thesis-code/data/processed/07_h2_normalized.csv")
print(df.shape)
print(df["region_type"].value_counts())

(19703, 5)
region_type
paragraph         19286
reference_list      417
Name: count, dtype: int64


#### 3. Assigning volume groups

In [7]:
import re

def get_issue_number(filepath):
    match = re.search(r"heresies_(\d+)_combined", filepath)
    return int(match.group(1)) if match else None

def get_volume(issue_nr):
    if issue_nr <= 4:
        return "Vol1_1977-1978"
    elif issue_nr <= 8:
        return "Vol2_1978-1979"
    elif issue_nr <= 12:
        return "Vol3_1980-1981"
    elif issue_nr <= 16:
        return "Vol4_1981-1983"
    elif issue_nr <= 20:
        return "Vol5_1984-1985"
    elif issue_nr <= 24:
        return "Vol6_1987-1989"
    elif issue_nr <= 27:
        return "Vol7_1990-1993"
    else:
        return None

df["issue"]  = df["source_file"].apply(get_issue_number)
df["volume"] = df["issue"].apply(get_volume)

print(df["volume"].value_counts())

volume
Vol2_1978-1979    4284
Vol1_1977-1978    3543
Vol3_1980-1981    3467
Vol4_1981-1983    2748
Vol6_1987-1989    2275
Vol5_1984-1985    2231
Vol7_1990-1993    1155
Name: count, dtype: int64


#### 4. Extracting person names
Using this: https://spacy.io/usage/linguistic-features#named-entities

In [8]:
def extract_persons(text):
    if pd.isna(text):
        return []
    
    doc = nlp(text)
    
    # only keep PERSON entities
    persons = [ent.text for ent in doc.ents if ent.label_ == "PERSON"]
    
    return persons

df["persons"] = df["text"].apply(extract_persons)
print("done")
print(df["persons"].iloc[0])

done
[]


#### 5. Checking what has been found
Using this: https://www.geeksforgeeks.org/python/python-counter-objects-elements/

In [9]:
# flatten all person names into one list
all_persons = [person for persons in df["persons"] for person in persons]

# count how often each name appears
from collections import Counter
person_counts = Counter(all_persons)

print(f"Total person mentions found: {len(all_persons)}")
print(f"Unique names found: {len(person_counts)}")
print("\nTop 30 most mentioned people:")
for name, count in person_counts.most_common(30):
    print(f"  {name}: {count}")

Total person mentions found: 2526
Unique names found: 1835

Top 30 most mentioned people:
  Martine: 26
  Donna: 23
  Vicki: 23
  Mia: 17
  Jean: 15
  Jane: 15
  Michelle: 13
  Ruth: 12
  CV: 12
  Margaret: 11
  Renée: 11
  Beth: 11
  Lacey: 11
  Lee: 11
  Edie: 10
  Alice: 10
  Norma: 9
  Michael: 9
  Martha: 9
  Chicanas: 8
  Ann: 8
  MA: 8
  Weber: 8
  Sarah: 8
  Floss: 8
  Ina: 7
  Ruthy: 6
  Mary: 6
  Myrna Zimmerman: 6
  Melvonna Ballenger: 6


In [10]:
def clean_persons(persons):
    cleaned = []
    for name in persons:
        # remove possessives
        name = name.replace("'s", "").strip()
        
        # skip single word names — too ambiguous
        if len(name.split()) < 2:
            continue
        
        # skip names with numbers or special characters
        if any(char.isdigit() for char in name):
            continue
            
        # skip short names under 5 characters total
       # if len(name) < 5:
        #    continue
            
        cleaned.append(name)
    return cleaned

df["persons"] = df["persons"].apply(clean_persons)

#### 6. Create one row per person mention with metadata

In [11]:
rows = []

for _, row in df.iterrows():
    for person in row["persons"]:
        rows.append({
            "source_file": row["source_file"],
            "issue":       row["issue"],
            "volume":      row["volume"],
            "region_type": row["region_type"],
            "person":      person
        })

df_persons = pd.DataFrame(rows)
print(df_persons.shape)
print(df_persons.head(30))

(1269, 5)
                                          source_file  issue          volume  \
0   /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_1978-1979   
1   /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_1978-1979   
2   /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_1978-1979   
3   /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_1978-1979   
4   /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_1978-1979   
5   /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_1978-1979   
6   /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_1978-1979   
7   /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_1978-1979   
8   /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_1978-1979   
9   /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_1978-1979   
10  /Users/sophiehamann/master-thesis-code/data/ra...      8  Vol2_1978-1979   
11  /Users/sophiehamann/master

In [12]:
# list of known false positives to remove
false_positives = {
    "Editorial Collective", "De Colores", "Sacred Song Sung",
    "Kitchen Dreams", "Kitchen Artists", "Kitchens", "Iris Films"
}

name_mapping = {
    "Ida b. Wells":     "Ida B. Wells-Barnett",
    "Ida B. Wells":     "Ida B. Wells-Barnett",
    "Lucy R. Lippard":  "Lucy Lippard",
}

def clean_name(name):
    # remove false positives
    if name in false_positives:
        return None
    # standardize duplicates
    if name in name_mapping:
        return name_mapping[name]
    return name

df_persons["person"] = df_persons["person"].apply(clean_name)
df_persons = df_persons.dropna(subset=["person"])

print(f"Cleaned person mentions: {len(df_persons)}")
print(df_persons["person"].value_counts().head(30))

Cleaned person mentions: 1262
person
Bessie Smith            7
Myrna Zimmerman         6
Melvonna Ballenger      6
Lucy Lippard            6
Virginia Woolf          5
Howardena Pindell       4
Calvin Klein            4
Sharon Larkin           4
Louis Sullivan          4
Natalie Barney          4
Suzanne Lacy            4
Ida B. Wells-Barnett    3
Sandra Maria Esteves    3
Barbara Ehrenreich      3
Toni Robertson          3
Mimi Smith              3
Adrienne Rich           3
Su Friedrich            3
Jean G. Facey           3
Spider Woman            3
Linda Nochlin           3
Frida Kahlo             3
Martha Wilson           3
Donna Grund Slepack     3
Nancy Spero             3
Cynthia Carr            3
Bernadette Devlin       3
Judy Chicago            3
Ruth Crawford           3
Esther Phillips         3
Name: count, dtype: int64


#### 7. Saving

In [13]:
df_persons.to_csv("/Users/sophiehamann/master-thesis-code/data/output_files/09_h2_persons.csv", index=False)